### SELF JOIN PRACTICE

In [6]:
import pandas as pd
import pandasql as psql

# Sample DataFrame
df = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5, 6, 7, 8, 9],
    'name': ['Alice', 'Bob', 'Charlie', 'Mike', 'Sam', 'Ram', 'Tom', 'Jerry', 'Ham'],
    'salary': [3000, 2000, 4000, 1000, 2500, 1200, 5000, 6000, 1800],
    'manager_id': [3, 7, 8, 3, 7, 3, 8, '', 7]
})

# SQL query
query = "SELECT * FROM df"

# Run query
result = psql.sqldf(query, locals())
print(result)


   emp_id     name  salary manager_id
0       1    Alice    3000          3
1       2      Bob    2000          7
2       3  Charlie    4000          8
3       4     Mike    1000          3
4       5      Sam    2500          7
5       6      Ram    1200          3
6       7      Tom    5000          8
7       8    Jerry    6000           
8       9      Ham    1800          7


In [10]:
# Find manager name

# SQL query
query = """

    SELECT d1.*, d2.name
    FROM df d1
    LEFT JOIN df d2
    ON d1.manager_id = d2.emp_id

"""

# Run query
result = psql.sqldf(query, locals())
print(result)

   emp_id     name  salary manager_id     name
0       1    Alice    3000          3  Charlie
1       2      Bob    2000          7      Tom
2       3  Charlie    4000          8    Jerry
3       4     Mike    1000          3  Charlie
4       5      Sam    2500          7      Tom
5       6      Ram    1200          3  Charlie
6       7      Tom    5000          8    Jerry
7       8    Jerry    6000                None
8       9      Ham    1800          7      Tom


In [11]:
# Find manager id whose employees has salary greater than 2000

# SQL query
query = """

    SELECT d1.*, d2.name
    FROM df d1
    LEFT JOIN df d2
    ON d1.manager_id = d2.emp_id
    WHERE d1.salary > 2000

"""

# Run query
result = psql.sqldf(query, locals())
print(result)

   emp_id     name  salary manager_id     name
0       1    Alice    3000          3  Charlie
1       3  Charlie    4000          8    Jerry
2       5      Sam    2500          7      Tom
3       7      Tom    5000          8    Jerry
4       8    Jerry    6000                None


In [13]:
# Find manager id whose employees has salary greater than 2000, but only emplyees

# SQL query
query = """

    SELECT d1.*, d2.name
    FROM df d1
    LEFT JOIN df d2
    ON d1.manager_id = d2.emp_id
    WHERE d1.salary > 2000 AND d1.emp_id NOT IN (SELECT manager_id FROM df)

"""

# Run query
result = psql.sqldf(query, locals())
print(result)

   emp_id   name  salary manager_id     name
0       1  Alice    3000          3  Charlie
1       5    Sam    2500          7      Tom


### USERS LOGGED FOR 3 CONSECUTIVE DAYS

In [94]:
import pandas as pd
import pandasql as psql

# Sample DataFrame
df = pd.DataFrame({
    'date': ['03/20/2021', '03/20/2021', '03/20/2021', '04/20/2021', '04/20/2021', '05/20/2021', '05/20/2021', '05/20/2021', '06/20/2021', '06/20/2021'],
    'name': ['Alice', 'Bob', 'Charlie', 'Alice', 'Bob', 'Alice', 'Bob', 'Charlie', 'Alice', 'Charlie']
})

# SQL query
query = "SELECT * FROM df"

# Run query
result = psql.sqldf(query, locals())
print(result)

         date     name
0  03/20/2021    Alice
1  03/20/2021      Bob
2  03/20/2021  Charlie
3  04/20/2021    Alice
4  04/20/2021      Bob
5  05/20/2021    Alice
6  05/20/2021      Bob
7  05/20/2021  Charlie
8  06/20/2021    Alice
9  06/20/2021  Charlie


In [95]:
# Users who logged in for 3 consecutive days

# SQL query
query = """

    WITH cte as (SELECT name, date,
    LEAD(date,2) OVER(PARTITION BY name) as op,
    LEAD(date,2) OVER(PARTITION BY name) - date as diff
    FROM df)

    SELECT DISTINCT(name) FROM cte
    WHERE diff = 2
    """

# Run query
result = psql.sqldf(query, locals())
print(result)

    name
0  Alice
1    Bob


In [106]:
# Users who logged in for 3 consecutive days

# SQL query
query = """


       WITH RankedLogins AS (
       SELECT
       name,
       date,
       ROW_NUMBER() OVER (PARTITION BY name ORDER BY date) AS rn
       FROM logins
        ),
        
       GroupedLogins AS (
          SELECT
            name,
            date,
            DATE_SUB(date, INTERVAL rn DAY) AS grp_key
          FROM RankedLogins
        ),

        
        ConsecutiveGroups AS (
          SELECT
            name,
            COUNT(*) AS consecutive_days,
            MIN(date) AS start_date,
            MAX(date) AS end_date
          FROM GroupedLogins
          GROUP BY name, grp_key
        )

        
        SELECT *
        FROM ConsecutiveGroups
        WHERE consecutive_days >= 3;

       
    
    """

# Run query
result = psql.sqldf(query, locals())
print(result)

      name        date  rn date_group
0    Alice  03/20/2021   1       None
1    Alice  04/20/2021   2       None
2    Alice  05/20/2021   3       None
3    Alice  06/20/2021   4       None
4      Bob  03/20/2021   1       None
5      Bob  04/20/2021   2       None
6      Bob  05/20/2021   3       None
7  Charlie  03/20/2021   1       None
8  Charlie  05/20/2021   2       None
9  Charlie  06/20/2021   3       None


In [92]:
import pandas as pd

# Sample data
data = {
    'name': ['Alice', 'Alice', 'Alice', 'Alice', 'Bob', 'Bob', 'Bob', 'Charlie', 'Charlie', 'Charlie'],
    'date': ['2023-08-01', '2023-08-02', '2023-08-03', '2023-08-05',
             '2023-08-01', '2023-08-02', '2023-08-04',
             '2023-08-01', '2023-08-02', '2023-08-03']
}

df = pd.DataFrame(data)
df['date'] = pd.to_datetime(df['date'])

# Remove duplicate logins on the same day per user
df_unique = df.drop_duplicates(subset=['name', 'date'])

# Rank dates per user
df['rn'] = df.groupby('name')['date'].rank(method='dense').astype(int)

# Create a group key by subtracting row number from date
df['grp_key'] = df['date'] - pd.to_timedelta(df['rn'], unit='D')

# Group by user and group key to find streaks
grouped = df.groupby(['name', 'grp_key']).agg(
    consecutive_days=('date', 'count'),
    start_date=('date', 'min'),
    end_date=('date', 'max')
).reset_index()

# Filter for streaks of at least 3 days
result = grouped[grouped['consecutive_days'] >= 3]
print(result)


      name    grp_key  consecutive_days start_date   end_date
0    Alice 2023-07-31                 3 2023-08-01 2023-08-03
4  Charlie 2023-07-31                 3 2023-08-01 2023-08-03
